1. Unpack graphs

In [1]:
import networkx as nx
import pandas as pd
import random
import numpy as np
import math
from cup_gather_data import hcgcr_data
import json
from tqdm import tqdm
from pathlib import Path
from unpack_graphs import unpack_graphs
from cup_gather_data import Iteration
from cup_main_faster import updateCR
from cup_functions_faster import update_queue_add_all_neighbors, \
    update_queue_remove_asymmetric, update_queue_remove_unchanged_orbits
    
base_path = Path(r'C:\Users\korol\OneDrive\Pulpit\Master\CUP - remove asymmetric, remove orbits, counting changes\test_graphs')
q_func_names = ["cor", "cas", "cup"]
update_functions = [
    update_queue_remove_unchanged_orbits,
    update_queue_remove_asymmetric,
    update_queue_add_all_neighbors,
]


In [2]:
def nxgraph(n, edges):
    G = nx.Graph()
    G.add_edges_from([tuple(e) for e in edges])
    G.add_nodes_from(range(int(n)))
    return G

def restore_dict_values(hash_dict):
    # hash_dict: {"hash": {"color": int, "orbit": list-or-set}, ...}
    output = {}
    for hash, val in hash_dict.items():
        color = int(val["color"])
        orbit = val.get("orbit")
        if isinstance(orbit, list):  # JSON-serialized set
            orbit = set(orbit)
        output[hash] = {"color": color, "orbit": orbit}
    return output

def unpack(name):
    df = pd.read_csv(base_path / name)
    # dataframe with two columns: G - the graph, iterations with list of iteration objects
    for col in ["graph", "coloring", "hash_dict"]:
        df[col] = df[col].apply(json.loads)
    graphs = []
    iterations = []
    for _, row in df.iterrows():
        graphs.append(nxgraph(row["n"], row["graph"]))
        iter = []
        for coloring_dict, hash_dict_from_json in zip(row["coloring"], row["hash_dict"]):
            coloring = pd.DataFrame(coloring_dict)
            hash_dict = restore_dict_values(hash_dict_from_json)
            iter.append(Iteration(coloring, hash_dict))
        iterations.append(iter)
    return pd.DataFrame({"graph": graphs, "iterations": iterations})

2. Generate updated graphs

In [3]:
def generate_G_up(G, S, alpha, seed=None):
    """
    Toggle edges only among nodes in the provided set S.

    - Uses p to choose k = ceil(p * C(|S|,2)) toggles.
    - Ensures every vertex in S appears in at least one toggled pair
    (requires k >= ceil(|S|/2); we automatically enforce that minimum).

    Parameters
    ----------
    G : networkx.Graph
    S : set (or any iterable)
        Affected vertices. Only edges with both endpoints in S may be toggled.
    alpha : float
        Fraction of possible internal pairs to toggle.
    seed : int | None
        Random seed for reproducibility.

    Returns
    -------
    G_new : networkx.Graph
    toggled_pairs : set[tuple]
        Set of toggled unordered pairs (u, v) with u < v.
    """
    rng = random.Random(seed)
    G_new = G.copy()

    S = set(S)
    s = len(S)
    if s < 2:
        return G_new, set()

    S_list = list(S)

    # Total possible internal pairs within S
    M = s * (s - 1) // 2

    # Number of toggles from alpha
    k = max(1, math.ceil(alpha * M))

    # Minimum toggles to ensure every vertex participates in at least one toggled pair
    k_min_cover = (s + 1) // 2
    k = max(k, k_min_cover)
    k = min(k, M)

    changed_pairs = set()

    # --- Step 1: coverage (pair up nodes so all appear at least once) ---
    S_mixed = S_list[:]
    rng.shuffle(S_mixed)

    for i in range(0, s - 1, 2):
        u, v = S_mixed[i], S_mixed[i + 1]
        a, b = (u, v) if u < v else (v, u)
        changed_pairs.add((a, b))

    if s % 2 == 1:
        last = S_mixed[-1]
        other = rng.choice(S_mixed[:-1])
        a, b = (last, other) if last < other else (other, last)
        changed_pairs.add((a, b))

    # --- Step 2: fill up to k with additional distinct internal pairs ---
    all_pairs = []
    for i in range(s):
        for j in range(i + 1, s):
            u, v = S_list[i], S_list[j]
            all_pairs.append((u, v) if u < v else (v, u))

    while len(changed_pairs) < k:
        changed_pairs.add(rng.choice(all_pairs))

    # --- Apply toggles ---
    for u, v in changed_pairs:
        if G_new.has_edge(u, v):
            G_new.remove_edge(u, v)
        else:
            G_new.add_edge(u, v)

    return G_new, k, changed_pairs

def draw_S(G, beta, remove_edge=False, seed=None):
    rng = random.Random(seed)
    if remove_edge:
        pair = rng.sample(list(G.edges()), 1)[0]
        return {pair[0], pair[1]}
    
    nodes = list(G.nodes())
    n = len(nodes)
    s = max(1, math.ceil(beta * n))
    S = set(rng.sample(nodes, s))
    return S

################ GENERATE UPDATED GRAPHS #####################

def random_graphs_up(df, S_list, alpha, seed=None):
    graphs_up = []
    for i, row in enumerate(tqdm(df.itertuples(index=False), desc="Generating for the file: ")):
        G = row.graph
        G_up, k, changed_edges = generate_G_up(G, S_list[i], alpha, seed)
        graphs_up.append({"graph_up": G_up, "S": S_list[i], "k": k, "changed_edges": changed_edges})
    return graphs_up

def update_dict(G, S, T_old, k, changed_edges, colorings_up, q_lens):
    avg = (sum(q_lens) / len(q_lens)) if q_lens else 0.0
    data = {
        "n": G.number_of_nodes(),
        "m": G.number_of_edges(),
        "k": k, # number of changed edges
        "T_old": T_old,
        "T_up": sum(x != 0 for x in q_lens),
        "q_lens": json.dumps(q_lens),
        "q_len_average": avg,
        "changed_edges": json.dumps(list(changed_edges)),
        "S": json.dumps(list(S)),
        "coloring": json.dumps(colorings_up)

    }
    return data

def main_func(df, graphs_up, q_func, q_func_name, path: Path, fname):
    path.mkdir(parents=True, exist_ok=True)
    result = []
    for i, row in enumerate(tqdm(df.itertuples(index=False), desc=f"{q_func_name}")):
        G_up = graphs_up[i]["graph_up"]
        S    = graphs_up[i]["S"]
        k    = graphs_up[i]["k"] # number of edges changed
        
        changed_edges = graphs_up[i]["changed_edges"]
        iterations = row.iterations
        T_old = len(iterations) + 1 # include in the iterations check step
        try:
            colorings_up, q_lens = updateCR(G_up, S, iterations, q_func)
            d = update_dict(G_up, S, T_old, k, changed_edges, colorings_up, q_lens)
        except Exception as e:
            print(f"Error in {q_func_name}, row {i}: {e},"
                f"{colorings_up}, {q_lens}")
            d = {        
                "n": G_up.number_of_nodes(),
                "m": G_up.number_of_edges(),
                "k": k, # number of changed edges
                "T_old":T_old,
                "T_up": json.dumps([]),
                "S": json.dumps(list(S)),
                "changed_edges": json.dumps(list(changed_edges)),
                "q_lens": json.dumps([]),
                "q_len_average": 0,
                "coloring": json.dumps([])
            }
        result.append(d)
        pd.DataFrame(result).to_csv(path / f"result_{fname}_{q_func_name}.csv", index=False)


In [4]:
# remove one edge in a tree
df = unpack("random_trees.csv")

# draw deleted edges
S_list = [
    draw_S(G=G, beta=None, remove_edge=True, seed="0000")
    for G in df["graph"]
]
# remove edges in copies of G
graphs_up = []
for (u, v), G in zip(S_list, df["graph"]):
    G_up = G.copy()
    G_up.remove_edge(u, v)
    graphs_up.append({"graph_up": G_up, "S": {u,v}, "k": 1, "changed_edges": [(u,v)]})

for qfn, q_func in zip(q_func_names, update_functions):
    print("We go with the update function:", qfn)
    try:
        main_func(df, graphs_up, q_func, qfn, base_path, "trees_remove_edge")
    except Exception as e:
        print("Error:", e)

We go with the update function: cor


cor: 250it [01:20,  3.10it/s]


We go with the update function: cas


cas: 250it [01:19,  3.16it/s]


We go with the update function: cup


cup: 250it [01:25,  2.94it/s]


## 2. choose randomly a percent of edges and change

In [5]:
def generate_perturbed_graph(G, p, seed=None):
    rng = random.Random(seed)
    Gp = G.copy()

    m = G.number_of_edges()
    k = max(1, int(p * m))  # number of edits
    k_del = k // 2
    k_add = k - k_del

    # --- Delete edges ---
    edges = list(Gp.edges())
    del_edges = []
    if k_del > 0:
        del_edges = rng.sample(edges, min(k_del, len(edges)))
        Gp.remove_edges_from(del_edges)

    # --- Add edges ---
    # Candidate non-edges
    non_edges = list(nx.non_edges(Gp))
    add_edges = []
    if k_add > 0 and non_edges:
        add_edges = rng.sample(non_edges, min(k_add, len(non_edges)))
        Gp.add_edges_from(add_edges)

    return Gp, (add_edges + del_edges)


In [6]:
# random graph with 0.03 edges
for f in ["05", "1", "3", "5"]:
    f_n = f"rg_p_{f}.csv"
    df = unpack(f_n)
    print(f"Random graphs from a file: {f_n}")

    for a in [0.001, 0.005, 0.01]:
        graphs_up = []
        for G in df["graph"]:
            G_up, changed_edges = generate_perturbed_graph(G, a, "1111")
            graphs_up.append({
                "graph_up": G_up, 
                "S": {v for e in changed_edges for v in e}, 
                "k": len(changed_edges), 
                "changed_edges": changed_edges})

        for qfn, q_func in zip(q_func_names, update_functions):
            print(f"a = {a}, we go with the update function: {qfn}")
            try:
                main_func(df, graphs_up, q_func, qfn, base_path, f"rg_p_{f}_a_{int(1000*a)}")
            except Exception as e:
                print("Error:", e)

Random graphs from a file: rg_p_05.csv
a = 0.001, we go with the update function: cor


cor: 250it [01:10,  3.55it/s]


a = 0.001, we go with the update function: cas


cas: 250it [01:10,  3.56it/s]


a = 0.001, we go with the update function: cup


...

cor: 250it [00:51,  4.86it/s]


a = 0.01, we go with the update function: cas


cas: 250it [00:52,  4.74it/s]


a = 0.01, we go with the update function: cup


cup: 250it [00:56,  4.41it/s]


### Add new columns
-> how many vertices changed color
-> total work (sum of q_len)
-> q_average without 0
-> delta T

In [7]:
def add_columns(f_name, r_name):
    df = pd.read_csv(base_path / f_name)
    rf = pd.read_csv(base_path / r_name)

    # How many vertices changed color
    col_main = df["coloring"].apply(json.loads)    # list[dict]
    col_ref  = rf["coloring"].apply(json.loads)    # list[list]

    # extract final colorings
    main_val = col_main.apply(lambda s: s[-1]["color"] if s else [])
    ref_val  = col_ref.apply(lambda s: s[-1]          if s else [])

    # hamming distance
    def hamming(a, b):
        return sum(x != y for x, y in zip(a, b))

    v_changed = main_val.combine(ref_val, hamming).astype(int)

    rf.insert(7, "v_changed", v_changed)
    
    # Total processed vertices
    col_main = rf["q_lens"].apply(json.loads) 
    col_main.apply(lambda lst: [int(v) for v in lst]) 
    
    total_work = col_main.apply(sum)
    rf.insert(8, "W", total_work)
    
    # Average q length without 0's
    col_main = rf["W"]
    col_help = rf["T_up"]
    q_average_real = (col_main / col_help).where(col_help != 0, None).astype(float)
    rf.insert(9, "q_avg_real", q_average_real)
    
    # Delta T
    col_main = rf["T_old"]
    col_help = rf["T_up"]
    delta_T = col_main - col_help
    rf.insert(5, "T_delta", delta_T)
    
    rf.to_csv(base_path / r_name)

In [8]:
for f in ["05", "1", "3", "5"]:
    f_n = f"rg_p_{f}.csv"
    for a in [0.001, 0.005, 0.01]:
        for qfn in q_func_names:
            r_n = f"result_rg_p_{f}_a_{int(1000*a)}_{qfn}.csv"
            add_columns(f_n, r_n)

In [9]:
for qfn in q_func_names:
    r_n = f"result_trees_remove_edge_{qfn}.csv"
    add_columns("random_trees.csv", r_n)

In [10]:
def add_symmetry_info_cols(f_name, r_name):
    df = pd.read_csv(base_path / f_name)
    rf = pd.read_csv(base_path / r_name)

    # How many vertices changed color
    col_main = df["coloring"].apply(json.loads)    # list[dict]
    col_ref  = rf["coloring"].apply(json.loads)    # list[list]

    # extract final colorings
    old_coloring = col_main.apply(lambda s: s[-1]["color"] if s else [])
    new_coloring  = col_ref.apply(lambda s: s[-1]          if s else [])
    
    def nr_symmetric_vertices(coloring):
        col = np.array(coloring)
        u, counts = np.unique(col, return_counts=True)
        symm = (counts > 1).nonzero() # indices of symmetric vertices
        nr_symm = sum(counts[symm])
        nr_unique = len(u)
        return int(nr_symm), int(nr_unique)
    
    # compute pairs
    sym_old_pairs = old_coloring.apply(nr_symmetric_vertices)
    sym_up_pairs  = new_coloring.apply(nr_symmetric_vertices)

    # unpack
    sym_old    = sym_old_pairs.apply(lambda p: p[0])
    unique_old = sym_old_pairs.apply(lambda p: p[1])
    sym_up     = sym_up_pairs.apply(lambda p: p[0])
    unique_up  = sym_up_pairs.apply(lambda p: p[1])

    # insert in order
    pos = 12  # choose your index
    rf.insert(pos,     "sym_old",    sym_old)
    rf.insert(pos + 1, "sym_up",     sym_up)
    rf.insert(pos + 2, "unique_old", unique_old)
    rf.insert(pos + 3, "unique_up",  unique_up)
    
    rf.to_csv(base_path / r_name)

In [11]:
for f in ["05", "1", "3", "5"]:
    f_n = f"rg_p_{f}.csv"
    for a in [0.001, 0.005, 0.01]:
        for qfn in q_func_names:
            r_n = f"result_rg_p_{f}_a_{int(1000*a)}_{qfn}.csv"
            add_symmetry_info_cols(f_n, r_n)
            
for qfn in q_func_names:
    r_n = f"result_trees_remove_edge_{qfn}.csv"
    add_symmetry_info_cols("random_trees.csv", r_n)